In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

DIR_FIGS = Path("/path/to/escala-6x1/outputs/figures")
DIR_TABS = Path("/path/to/escala-6x1/outputs/tables")

plt.rcParams.update({
    "figure.dpi": 150,
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
})

print("Imports OK")

In [ ]:
PIB_2023    = 10.9e12   
DELTA_CNI   = -0.007    

PARCELA_AFETADA = 0.194

print(f"Cenário base CNI:       {DELTA_CNI*100:.1f}%  =  R$ {PIB_2023*DELTA_CNI/1e9:.0f} bi")
print(f"Parcela afetada (PNAD): {PARCELA_AFETADA*100:.1f}% dos ocupados")

In [ ]:
cenarios = [
    {
        "nome": "Modelo CNI\n(sem ganho produtividade)",
        "ganho_produt": 0.00,
        "cor": "#E84545",
        "fonte": "CNI (2026)"
    },
    {
        "nome": "Ganho moderado\n(+3% produt./hora)",
        "ganho_produt": 0.03,
        "cor": "#F5A623",
        "fonte": "Estimativa conservadora"
    },
    {
        "nome": "Ganho intermediário\n(+6% produt./hora)",
        "ganho_produt": 0.06,
        "cor": "#2C6FBF",
        "fonte": "Collewet & Sauermann (2017)"
    },
    {
        "nome": "Ganho alto\n(+9% produt./hora)",
        "ganho_produt": 0.09,
        "cor": "#27AE60",
        "fonte": "Pencavel (2015) — limite superior"
    },
]

for c in cenarios:
    
    compensacao   = c["ganho_produt"] * PARCELA_AFETADA
    c["delta_pib"] = DELTA_CNI + compensacao
    c["impacto_bi"] = PIB_2023 * c["delta_pib"] / 1e9

print("\n── Tabela de cenários ──────────────────────────────────────")
print(f"{'Cenário':<40} {'Δ PIB':>7} {'R$ bi':>10}")
print("-" * 60)
for c in cenarios:
    print(f"{c['nome'].replace(chr(10),' '):<40} {c['delta_pib']*100:>6.2f}%  {c['impacto_bi']:>9.1f}")

In [ ]:
df_out = pd.DataFrame([{
    "cenario": c["nome"].replace("\n", " "),
    "ganho_produtividade_pct": c["ganho_produt"] * 100,
    "delta_pib_pct": round(c["delta_pib"] * 100, 3),
    "impacto_r$_bilhoes": round(c["impacto_bi"], 1),
    "fonte": c["fonte"],
} for c in cenarios])

df_out.to_csv(DIR_TABS / "cenarios_impacto_pib.csv", index=False)
print("\nTabela salva.")
print(df_out.to_string(index=False))

In [ ]:
nomes   = [c["nome"] for c in cenarios]
valores = [c["delta_pib"] * 100 for c in cenarios]
cores   = [c["cor"] for c in cenarios]
bilhoes = [c["impacto_bi"] for c in cenarios]

fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.bar(nomes, valores, color=cores, width=0.5, edgecolor="white")
ax.axhline(0, color="black", linewidth=0.8)

for bar, val, bi in zip(bars, valores, bilhoes):
    if val < 0:
        ypos = val / 2  
        cor_txt = "white"
    else:
        ypos = val + 0.02
        cor_txt = "black"
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        ypos,
        f"{val:+.2f}%\nR$ {bi:+.0f} bi",
        ha="center", va="center", fontsize=10,
        fontweight="bold", color=cor_txt
    )

ax.set_ylabel("Variação estimada no PIB (%)", fontsize=11)
ax.set_title(
    "Impacto no PIB da redução de jornada (44h→40h) — cenários comparados\n"
    "Âncora: projeção CNI (-0,7%). Ganhos de produtividade: literatura internacional.",
    fontsize=12, loc="left"
)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100, decimals=2))
ax.set_ylim(min(valores) * 1.3, 0.3)

fig.text(
    0.01, -0.02,
    "Metodologia: cenário CNI = âncora de -0,7% (CNI, 2026). "
    "Compensação = ganho produtividade/hora × parcela afetada (19,4% dos ocupados, PNAD 2023). "
    "Limitação: não captura efeitos de demanda agregada nem transição setorial.",
    fontsize=7.5, color="gray", transform=fig.transFigure
)

plt.tight_layout()
plt.savefig(DIR_FIGS / "04_cenarios_impacto_pib.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico 4 salvo.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

componentes = ["Efeito-volume\n(↓ horas)", "Efeito-produtividade\n(↑ produt./hora)", "Efeito líquido\n(resultado final)"]

cni_vol  = DELTA_CNI * 100
cni_prod = 0.0
cni_net  = cni_vol + cni_prod

alt_vol  = DELTA_CNI * 100
alt_prod = 0.06 * PARCELA_AFETADA * 100   
alt_net  = alt_vol + alt_prod

x = np.arange(len(componentes))
w = 0.3

b1 = ax.bar(x - w/2, [cni_vol, cni_prod, cni_net], width=w,
            color=["#E84545", "#DDDDDD", "#E84545"],
            label="Modelo CNI", edgecolor="white")

b2 = ax.bar(x + w/2, [alt_vol, alt_prod, alt_net], width=w,
            color=["#E84545", "#27AE60", "#2C6FBF"],
            label="Cenário +6% produt./hora", edgecolor="white")

ax.axhline(0, color="black", linewidth=0.8)

for bar in list(b1) + list(b2):
    val = bar.get_height()
    if abs(val) > 0.01:
        offset = 0.03 if val >= 0 else -0.06
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            val + offset,
            f"{val:+.2f}%",
            ha="center", fontsize=9
        )

ax.set_xticks(x)
ax.set_xticklabels(componentes, fontsize=7)
ax.set_ylabel("Contribuição para Δ PIB (%)", fontsize=7)
ax.set_title(
    "Decomposição do impacto: o que o modelo da CNI ignora\n"
    "Efeito-volume vs. efeito-produtividade (parcela afetada: 19,4% dos ocupados)",
    fontsize=8, loc="left"
)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100, decimals=2))
ax.legend(frameon=False, fontsize=10)
ax.set_ylim(min(cni_vol, alt_vol) * 1.3, max(alt_prod, 0.3))

plt.tight_layout()
plt.savefig(DIR_FIGS / "05_decomposicao_efeito_volume_produtividade.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico 5 salvo.")